<a href="https://colab.research.google.com/github/Aries2003/raiseLab/blob/main/llama3_2_1b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ollama

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.7 MB/s eta 0:00:00


# **Llama QA (llama3.2:1b)**

In [1]:
# from operator import index
import os
import pandas as pd
import ollama

"""ARC_DA Dataset Format"""

DATA_PATH = r"/home/valmeid1/raiseLab/dataset/combined_arc.parquet"
OUTPUT_PATH = r"/home/valmeid1/raiseLab/llm_output/llama/QA.parquet"

MODEL_NAME = "llama3.2:1b"

def prompt_builder(row):
    base_question = row["question"]
    choices_dict = row["choices"]

    labels = choices_dict.get('label', [])
    texts = choices_dict.get('text', [])

    options_text = ''

    for label, text in zip(labels, texts):
        options_text += f"{label}){text}\n"

    prompt =    f"Question: {base_question}\n\nChoices:\n{options_text}"
    # print(prompt)
    return prompt


def query_model(formatted_prompt):

    system_instruction = """You are an expert science assistant taking a multiple-choice exam.\n
        Analyze the question and choices provided. Respond with EXACTLY One
        sentence representing the correct answer choice.\n
        Do NOT write punctuation, or extra spaces."""

    response = ollama.chat(

        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": formatted_prompt}
        ],
        options=
            {"temperature": 0.0,
            # "num_predict":90
        }
    )
    # print(response)
    return response['message']['content'].strip()

def main():
    if not os.path.exists(DATA_PATH):
        print(f"Error: Dataset not found at {DATA_PATH}")
        return

    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {len(df)} rows. Starting evaluation via Ollama ({MODEL_NAME})...")

    answers = []

    for index, row in df.iterrows():
        prompt = prompt_builder(row)

        if index % 10 == 0:
            print(f"Processing row {index}/{len(df)}...")

        try:
            answer = query_model(prompt)
            answers.append(answer)
        except Exception as e:
            print(f"Error at index {index}: {e}")
            answers.append("ERROR: Ollama generation failed")
    df['model_answer'] = answers
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Done! Results successfully saved to {OUTPUT_PATH}")

def check_output():

    df = pd.read_parquet(OUTPUT_PATH)
    index =19
    print(df.question[index])
    print(df.model_answer[index])
    # print(df.iloc[index, 1:4])

def single_query_run():
    df = pd.read_parquet(DATA_PATH)
    index = 11
    row = df.iloc[index]
    p = prompt_builder(row)
    try:
        answer = query_model(p)

        print(df.question[index])

        print(answer)
    except Exception as e:
        print(f"Error at index : {e}")

if __name__ == "__main__":
    main()
    # single_query_run()
    # check_output()

Loaded 7787 rows. Starting evaluation via Ollama (llama3.2:1b)...
Processing row 0/7787...
Processing row 10/7787...
Processing row 20/7787...
Processing row 30/7787...
Processing row 40/7787...
Processing row 50/7787...
Processing row 60/7787...
Processing row 70/7787...
Processing row 80/7787...
Processing row 90/7787...
Processing row 100/7787...
Processing row 110/7787...
Processing row 120/7787...
Processing row 130/7787...
Processing row 140/7787...
Processing row 150/7787...
Processing row 160/7787...
Processing row 170/7787...
Processing row 180/7787...
Processing row 190/7787...
Processing row 200/7787...
Processing row 210/7787...
Processing row 220/7787...
Processing row 230/7787...
Processing row 240/7787...
Processing row 250/7787...
Processing row 260/7787...
Processing row 270/7787...
Processing row 280/7787...
Processing row 290/7787...
Processing row 300/7787...
Processing row 310/7787...
Processing row 320/7787...
Processing row 330/7787...
Processing row 340/7787...


# **Llama_Summary**

In [1]:
import pandas as pd

train_file = "/home/valmeid1/raiseLab/dataset/summary_dataset/train-00000-of-00001.parquet"
test_file = "/home/valmeid1/raiseLab/dataset/summary_dataset/test-00000-of-00001.parquet"
validation_file = "/home/valmeid1/raiseLab/dataset/summary_dataset/validation-00000-of-00001.parquet"

df_train = pd.read_parquet(train_file)
df_validation = pd.read_parquet(validation_file)
df_test = pd.read_parquet(test_file)

print(f"Train DataFrame shape: {df_train.shape}")
print(f"Validation DataFrame shape: {df_validation.shape}")
print(f"Test DataFrame shape: {df_test.shape}")
combined_df = pd.concat([df_train, df_validation], ignore_index=True)
combined_df = pd.concat([combined_df, df_test], ignore_index=True)
print(f"Combined DataFrame shape: {combined_df.shape}")
# You can uncomment the line below to display the first few rows of the combined DataFrame
print(combined_df.head())

Train DataFrame shape: (204045, 3)
Validation DataFrame shape: (11332, 3)
Test DataFrame shape: (11334, 3)
Combined DataFrame shape: (226711, 3)
                                            document  \
0  The full cost of damage in Newton Stewart, one...   
1  A fire alarm went off at the Holiday Inn in Ho...   
2  Ferrari appeared in a position to challenge un...   
3  John Edward Bates, formerly of Spalding, Linco...   
4  Patients and staff were evacuated from Cerahpa...   

                                             summary        id  
0  Clean-up operations are continuing across the ...  35232142  
1  Two tourist buses have been destroyed by fire ...  40143035  
2  Lewis Hamilton stormed to pole position at the...  35951548  
3  A former Lincolnshire Police officer carried o...  36266422  
4  An armed man who locked himself into a room at...  38826984  


In [ ]:
import pandas as pd
import ollama
import os

# DATA_PATH = r"dataset\XSUM\validation-00000-of-00001 (1).parquet"
DATA_PATH = combined_df
OUTPUT_PATH = r"output\Summarization\llama_summary.parquet"

MODEL_NAME = "llama3.2:1b"

def prompt_builder(row):
    article_text = row["document"]
    summ = row["summary"]
    id = row["id"]


    prompt = (
        f"Context Article: \n {article_text}\n\n"
        f"Provide a single sentence, highly concise summary of the article above."
    )

    return prompt

def query_model(formatted_prompt ):

    system_instruction = ("""You are expert journalist tasked with extreme summurization. \n
                            Read the provided article and summarize it in exactly ONE concise sentence\n.
                            Do not include introductory phrasing like 'Here is a summary:' or 'This article discusses'.
                            Output only the final summary sentence directly""")

    response = ollama.chat(
        model= MODEL_NAME,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": formatted_prompt}

        ],
        options={
            "temperature": 0.0
            }
    )

    return response['message']['content'].strip()

def main():
    # if not os.path.exists(DATA_PATH):
    #     print(f"Error: Dataset doesnt exist")

    #     return

    # df = pd.read_parquet(DATA_PATH)
    df =DATA_PATH
    print(f"Dataset loaded {len(df)}")

    model_summaries = []

    for index, row in df.iterrows():

        prompt = prompt_builder(row)

        if index % 50 ==0:
            print(f"Evaluating instance {index}/ {len(df)}.....")

        try:
            generatd_summaries = query_model(prompt)
            model_summaries.append(generatd_summaries)
        except Exception as e:
            print(f"ERROR at indexx {index}: {e}")
            model_summaries.append("ERROR: Generation Failed")


    df['model_summary'] = model_summaries
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Evaluation Complete! Results at {OUTPUT_PATH}")

def check_output():

    # df = pd.read_parquet(OUTPUT_PATH)
    df = DATA_PATH
    index =19
    row = df.iloc[index]
    print(f'\n Article: {row["document"]}\n')
    print(f'Summary: {row["summary"]}\n')
    print(f'Model Generated: {row['model_summary']}')
    # # print(df.iloc[index, 1:4])

def single_query_run():
    # df = pd.read_parquet(DATA_PATH)
    df =DATA_PATH
    index =19
    p = prompt_builder(df.iloc[index])
    try:
        generated_summaries = query_model(p)

        print(f"Article:\n {df.document[index]}\n")
        print(f"Summary: {df.summary[index]}\n")
        print(f"Model Generated: {generated_summaries}")
    except Exception as e:
        print(f"Error at index : {e}")

if __name__ == "__main__":
    main()
    # single_query_run()
    # check_output()

Dataset loaded 226711
Evaluating instance 0/ 226711.....
Evaluating instance 50/ 226711.....
Evaluating instance 100/ 226711.....
Evaluating instance 150/ 226711.....
Evaluating instance 200/ 226711.....
Evaluating instance 250/ 226711.....
Evaluating instance 300/ 226711.....
Evaluating instance 350/ 226711.....
Evaluating instance 400/ 226711.....
Evaluating instance 450/ 226711.....
Evaluating instance 500/ 226711.....
Evaluating instance 550/ 226711.....
Evaluating instance 600/ 226711.....
Evaluating instance 650/ 226711.....
Evaluating instance 700/ 226711.....
Evaluating instance 750/ 226711.....
Evaluating instance 800/ 226711.....
Evaluating instance 850/ 226711.....
Evaluating instance 900/ 226711.....
Evaluating instance 950/ 226711.....
Evaluating instance 1000/ 226711.....
Evaluating instance 1050/ 226711.....
Evaluating instance 1100/ 226711.....
Evaluating instance 1150/ 226711.....
Evaluating instance 1200/ 226711.....
Evaluating instance 1250/ 226711.....
Evaluating in

# **Llama Reasoning (llama3.2:1b)**

In [ ]:
import pandas as pd

train_file = "/home/valmeid1/raiseLab/dataset/reason_dataset/train-00000-of-00001.parquet"
validation_file = "/home/valmeid1/raiseLab/dataset/reason_dataset/validation-00000-of-00001.parquet"

df_train = pd.read_parquet(train_file)
df_validation = pd.read_parquet(validation_file)

combined_df = pd.concat([df_train, df_validation], ignore_index=True)

print(f"Combined DataFrame shape: {combined_df.shape}")
# You can uncomment the line below to display the first few rows of the combined DataFrame
print(combined_df.head())

In [ ]:

import os
import pandas as pd
import ollama




# DATA_PATH = r"dataset\NLG\validation-00000-of-00001.parquet"
DATA_PATH = combined_df
OUTPUT_PATH= r"output\Reasoning\llama_reasoning.parquet"

MODEL_NAME = "llama3.2:1b"


def prompt_builder(row):
    obs1 = row["observation_1"]
    obs2 = row["observation_2"]
    h1 = row["hypothesis_1"]
    h2 = row["hypothesis_2"]

    prompt = (
        f"A narrative timeline has a beggining and an end , but is missing the middle event \n\n"
        f"Beginning (Observation 1): {obs1} \n"
        f"Ending (Observation 2): {obs2} \n\n"
        f"Which of the following hypothesis is the most plausible middle event that connects them \n"
        f"1) {h1}\n"
        f"2) {h2}\n"
    )
    # print(prompt)
    return prompt,h1,h2

def query_model(formatted_prompt):

    system_instruction = """You are an expert in logic and commonsense reasoning . \n\n                          Analyze the provided observation and select the most plausible hypothesis.\n\n                          Respond with EXACTLY the digit '1' or '2' corresponding to the correct hypothesis \n\n                          Do NOT write any introduction, punctuation, or explainations. Output only a single number character """

    response = ollama.chat(
        model = MODEL_NAME,
        messages= [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": formatted_prompt}
        ],
        options={
            "temperature": 0.0,
            # "num_predict":5
        }

    )

    return response['message']['content'].strip()

def main():
    # The os.path.exists check is no longer valid if DATA_PATH is a DataFrame
    # if not os.path.exists(DATA_PATH):
    #     print(f"Error: Dataset not found at {DATA_PATH}")
    #     return

    # df = pd.read_parquet(DATA_PATH)
    df = DATA_PATH # DATA_PATH is already combined_df (a DataFrame)
    print(f"Loaded {len(df)} dataset instance. Starting Evaluation")

    model_answers = []

    for index, row in df.iterrows():

        prompt, h1,h2 = prompt_builder(row)

        if index % 50 == 0:

            print(f"Evaluating instance {index}/{len(df)}...")

        try:
            ans= query_model(prompt)

            if '1' in ans:
                model_answers.append(h1)
            elif '2' in ans:
                model_answers.append(h2)
            else:
                model_answers.append(-1)
        except Exception as e:
            print(f"Error at index {index}: {e}")
            model_answers.append(-1)

    df['model_answer'] = model_answers
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Evaluation complete! Results exported to {OUTPUT_PATH}")

def check_output():

    df = pd.read_parquet(OUTPUT_PATH)
    index =19
    row = df.iloc[index]
    print(f'\nObservation 1: {row["observation_1"]}')
    print(f'Observation 2: {row["observation_2"]}')
    print(f'\n Hypo 1: {row["hypothesis_1"]}')
    print(f'Hypo 2: {row["hypothesis_2"]}')
    print(f'Answer: {row["label"]}')
    print(f'Model selected: {row["model_answer"]}')
    # print(df.iloc[index, 1:4])

def single_query_run():

    index = 11
    # df = pd.read_parquet(DATA_PATH) # This line caused the TypeError
    df = DATA_PATH # Correctly use the DataFrame already assigned to DATA_PATH
    p, hypothesis_1, hypothesis_2 = prompt_builder(df.iloc[index])

    try:
        ans = query_model(p)
        print(p)

        if '1' in ans:
            print(f"Model chose 1: {hypothesis_1}")
        elif '2' in ans:
            print(f"Model chose 2: {hypothesis_2}")
        else:
            print(f"Malformed model response: {ans}")
    except Exception as e:
        print(f"Error at index {index}: {e}")

if __name__ == "__main__":
    main()
    # single_query_run()
    # check_output()
